# 07 - Evaluate ECG-CXR bridge retrieval

Primary validation outcomes are ECG→CXR and CXR→ECG all-to-all retrieval. The `val_retrieval` and `test` splits additionally evaluate one true same-admission match among ten candidates. These metrics measure encounter association, not diagnostic prediction.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/data/liangz2/ecg_analysis/symile/ecg_cxr_bridge_artifacts",
))
FEATURE_ROOT = ARTIFACT_ROOT / "features"
CODE_ROOT = ARTIFACT_ROOT / "code"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_codebind
importlib.reload(ecg_cxr_codebind)
RUN_NAME = os.environ.get("ECG_CXR_RUN_NAME", "symile_ecg_cxr_bridge_v1")
RUN_DIR = ARTIFACT_ROOT / "models" / RUN_NAME
CHECKPOINT = RUN_DIR / "best.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = int(os.environ.get("ECG_CXR_EVAL_BATCH_SIZE", "512"))
print({"checkpoint": str(CHECKPOINT), "device": str(DEVICE)})


In [ ]:
class EvaluationDataset(Dataset):
    def __init__(self, split):
        self.split = split
        self.arrays = {"ecg": np.load(FEATURE_ROOT / split / "ecg_hubert.npy", mmap_mode="r"),
                       "cxr": np.load(FEATURE_ROOT / split / "cxr_raddino.npy", mmap_mode="r")}
        self.manifest = [json.loads(line) for line in
                         (ARTIFACT_ROOT / "manifests" / f"{split}.jsonl").read_text().splitlines()]
        assert all(len(array) == len(self.manifest) for array in self.arrays.values())

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest[index]
        item = {name: torch.from_numpy(np.array(array[index], dtype="float32", copy=True))
                for name, array in self.arrays.items()}
        for key in ["subject_id", "hadm_id", "label_hadm_id", "label"]:
            if key in row:
                item[key] = torch.tensor(row[key], dtype=torch.long)
        return item

checkpoint = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
resolved = checkpoint["resolved_config"]
model = ecg_cxr_codebind.ECGCXRCodeBind(
    resolved["input_dims"], **resolved["model_hparams"]
).to(DEVICE).eval().requires_grad_(False)
model.load_state_dict(checkpoint["model"])
print("Loaded epoch:", checkpoint["epoch"])


In [ ]:
@torch.inference_mode()
def encode_split(split):
    dataset = EvaluationDataset(split)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                        pin_memory=DEVICE.type == "cuda")
    collected = {}
    for batch in loader:
        outputs = model({name: batch[name].to(DEVICE, non_blocking=True)
                         for name in ecg_cxr_codebind.MODALITIES})
        values = {}
        for name, output in outputs.items():
            values[name] = output.common.cpu()
            values[name + "_specific"] = output.specific.cpu()
            values[name + "_common_indices"] = output.common_indices.cpu()
            values[name + "_specific_indices"] = output.specific_indices.cpu()
        for key in ["subject_id", "hadm_id", "label_hadm_id", "label"]:
            if key in batch:
                values[key] = batch[key]
        for key, value in values.items():
            collected.setdefault(key, []).append(value)
    return dataset, {key: torch.cat(parts) for key, parts in collected.items()}

val_dataset, val = encode_split("val")
val_retrieval_dataset, val_retrieval = encode_split("val_retrieval")
test_dataset, test = encode_split("test")
print({"val": len(val["hadm_id"]), "val_retrieval": len(val_retrieval["hadm_id"]),
       "test": len(test["hadm_id"])})


In [ ]:
def ranks_to_metrics(ranks, candidate_counts):
    ranks = torch.as_tensor(ranks, dtype=torch.float32)
    counts = torch.as_tensor(candidate_counts, dtype=torch.float32)
    return {"n_queries": int(len(ranks)),
            "recall_at_1": float((ranks <= 1).float().mean()),
            "recall_at_5": float((ranks <= 5).float().mean()),
            "recall_at_10": float((ranks <= 10).float().mean()),
            "mean_reciprocal_rank": float((1 / ranks).mean()),
            "median_rank": float(ranks.median()), "mean_rank": float(ranks.mean()),
            "random_chance_recall_at_1": float((1 / counts).mean())}

def all_to_all(query, candidate, subject_id):
    similarity = F.normalize(query.float(), dim=-1) @ F.normalize(candidate.float(), dim=-1).T
    same_patient = subject_id[:, None].eq(subject_id[None, :])
    off_diagonal = ~torch.eye(len(subject_id), dtype=torch.bool)
    similarity = similarity.masked_fill(same_patient & off_diagonal, -torch.inf)
    target = similarity.diag()
    ranks = 1 + (similarity > target[:, None]).sum(1)
    return ranks_to_metrics(ranks, torch.isfinite(similarity).sum(1))

def ten_candidate(encoded, query_name, candidate_name):
    query_hadm = encoded["label_hadm_id"]
    labels = encoded["label"]
    query_embedding = F.normalize(encoded[query_name].float(), dim=-1)
    candidate_embedding = F.normalize(encoded[candidate_name].float(), dim=-1)
    ranks = []
    counts = []
    for hadm_id in torch.unique(query_hadm):
        group = torch.where(query_hadm == hadm_id)[0]
        positives = group[labels[group] == 1]
        if len(group) != 10 or len(positives) != 1:
            raise ValueError(f"Invalid candidate group for {int(hadm_id)}")
        positive = positives[0]
        scores = candidate_embedding[group] @ query_embedding[positive]
        positive_position = torch.where(group == positive)[0][0]
        ranks.append(1 + int((scores > scores[positive_position]).sum()))
        counts.append(len(group))
    return ranks_to_metrics(ranks, counts)


In [ ]:
results = {
    "run_name": RUN_NAME, "checkpoint": str(CHECKPOINT),
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "validation_all_to_all": {
        "ecg_to_cxr": all_to_all(val["ecg"], val["cxr"], val["subject_id"]),
        "cxr_to_ecg": all_to_all(val["cxr"], val["ecg"], val["subject_id"]),
    },
    "validation_10_candidate": {
        "ecg_to_cxr": ten_candidate(val_retrieval, "ecg", "cxr"),
        "cxr_to_ecg": ten_candidate(val_retrieval, "cxr", "ecg"),
    },
    "test_10_candidate": {
        "ecg_to_cxr": ten_candidate(test, "ecg", "cxr"),
        "cxr_to_ecg": ten_candidate(test, "cxr", "ecg"),
    },
    "interpretation": ("Same-admission association retrieval only. The bridge was not trained "
        "with ECG diagnoses, CXR labels, reports, or laboratory values."),
}
display(pd.json_normalize(results).T)


In [ ]:
def utilization(encoded, suffix, size):
    return {name: {"used_codes": int(torch.unique(encoded[name + suffix]).numel()),
                   "total_codes": int(size),
                   "utilization": float(torch.unique(encoded[name + suffix]).numel() / size)}
            for name in ecg_cxr_codebind.MODALITIES}
results["codebook_diagnostics"] = {
    "common": utilization(val, "_common_indices", resolved["model_hparams"]["common_codebook_size"]),
    "specific": utilization(val, "_specific_indices", resolved["model_hparams"]["specific_codebook_size"]),
}
output = ARTIFACT_ROOT / "reports" / "07_ecg_cxr_bridge_metrics.json"
output.write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))
print("Saved:", output)
